In [50]:
import pandas as pd
import numpy as np
import joblib

np.random.seed(None)  # ensures different results every run
BASE_PATH = r'C:\Users\harsh\OneDrive\Desktop\ipl data analysis\datanew'
matches = pd.read_csv(rf'{BASE_PATH}\matches_clean.csv', parse_dates=['match_date'])

# Fix 2023 final — CSK won, not GT
matches.loc[matches['match_id'] == 1359544, 'match_winner'] = 'Chennai Super Kings'

# Fix 2024 final — KKR won, not SRH  
matches.loc[matches['match_id'] == 1426307, 'match_winner'] = 'Kolkata Knight Riders'
# Correct IPL winners by match_id
final_fixes = {
    336019:  'Rajasthan Royals',       # 2008
    392236:  'Deccan Chargers',         # 2009
    419161:  'Chennai Super Kings',     # 2010
    501267:  'Chennai Super Kings',     # 2011
    548377:  'Kolkata Knight Riders',   # 2012
    598069:  'Mumbai Indians',          # 2013
    734041:  'Kolkata Knight Riders',   # 2014
    829815:  'Mumbai Indians',          # 2015
    981011:  'Sunrisers Hyderabad',     # 2016
    1082646: 'Mumbai Indians',          # 2017
    1216495: 'Chennai Super Kings',     # 2021
    1304116: 'Gujarat Titans',          # 2022
}

for match_id, winner in final_fixes.items():
    matches.loc[matches['match_id'] == match_id, 'match_winner'] = winner

# Verify
finals_check = (matches.copy()
                .assign(match_number=pd.to_numeric(matches['match_number'], errors='coerce'))
                .groupby('season')
                .apply(lambda x: x.nlargest(1, 'match_number'), include_groups=False)
                .reset_index()
                [['season', 'match_winner']])
print(finals_check.sort_values('season').to_string())
# Verify
print(matches[matches['match_id'].isin([1359544, 1426307])][['match_id', 'season', 'team1', 'team2', 'match_winner']])

# ── 2026 Points Table ─────────────────────────────────────────────────────────
matches_2026 = matches[matches['season'] == 2026].copy()

t1 = matches_2026[['match_id', 'team1', 'match_winner']].copy()
t1.columns = ['match_id', 'team', 'match_winner']

t2 = matches_2026[['match_id', 'team2', 'match_winner']].copy()
t2.columns = ['match_id', 'team', 'match_winner']

all_2026 = pd.concat([t1, t2], ignore_index=True)
all_2026['won'] = (all_2026['team'] == all_2026['match_winner']).astype(int)

points_table = (all_2026.groupby('team')
                .agg(
                    matches=('won', 'count'),
                    wins=('won', 'sum')
                )
                .reset_index())

points_table['losses'] = points_table['matches'] - points_table['wins']
points_table['points'] = points_table['wins'] * 2
points_table = points_table.sort_values('points', ascending=False).reset_index(drop=True)
points_table['position'] = points_table.index + 1

print(points_table[['position', 'team', 'matches', 'wins', 'losses', 'points']])

    season                 match_winner
0     2008             Rajasthan Royals
1     2009              Deccan Chargers
2     2010          Chennai Super Kings
3     2011          Chennai Super Kings
4     2012        Kolkata Knight Riders
5     2013               Mumbai Indians
6     2014        Kolkata Knight Riders
7     2015               Mumbai Indians
8     2016          Sunrisers Hyderabad
9     2017               Mumbai Indians
10    2018          Chennai Super Kings
11    2019               Mumbai Indians
12    2021          Chennai Super Kings
13    2022               Gujarat Titans
14    2023          Chennai Super Kings
15    2024        Kolkata Knight Riders
16    2025  Royal Challengers Bangalore
17    2026               Delhi Capitals
     match_id  season                        team1                team2  \
443   1359544    2023  Royal Challengers Bangalore       Gujarat Titans   
514   1426307    2024                 Punjab Kings  Sunrisers Hyderabad   

              

In [51]:
# ── Recent form (2023-2025) ───────────────────────────────────────────────────
recent = matches[(matches['match_winner'].notna()) &
                 (matches['season'].isin([2023, 2024, 2025]))].copy()

r1 = recent[['team1', 'match_winner']].copy()
r1.columns = ['team', 'match_winner']
r2 = recent[['team2', 'match_winner']].copy()
r2.columns = ['team', 'match_winner']

all_recent = pd.concat([r1, r2])
all_recent['won'] = (all_recent['team'] == all_recent['match_winner']).astype(int)

recent_win_pct = (all_recent.groupby('team')['won']
                 .agg(['sum', 'count'])
                 .reset_index())
recent_win_pct.columns = ['team', 'wins_recent', 'matches_recent']
recent_win_pct['recent_win_pct'] = recent_win_pct['wins_recent'] / recent_win_pct['matches_recent']

# ── Historical strength (2008-2022) ──────────────────────────────────────────
historical = matches[(matches['match_winner'].notna()) &
                     (matches['season'] <= 2022)].copy()

h1 = historical[['team1', 'match_winner']].copy()
h1.columns = ['team', 'match_winner']
h2 = historical[['team2', 'match_winner']].copy()
h2.columns = ['team', 'match_winner']

all_hist = pd.concat([h1, h2])
all_hist['won'] = (all_hist['team'] == all_hist['match_winner']).astype(int)

hist_win_pct = (all_hist.groupby('team')['won']
                .agg(['sum', 'count'])
                .reset_index())
hist_win_pct.columns = ['team', 'wins_hist', 'matches_hist']
hist_win_pct['hist_win_pct'] = hist_win_pct['wins_hist'] / hist_win_pct['matches_hist']

# ── Active teams only ─────────────────────────────────────────────────────────
active_teams = ['Chennai Super Kings', 'Mumbai Indians', 'Royal Challengers Bangalore',
                'Kolkata Knight Riders', 'Rajasthan Royals', 'Sunrisers Hyderabad',
                'Delhi Capitals', 'Punjab Kings', 'Gujarat Titans', 'Lucknow Super Giants']

# Build strength table
strength_table = pd.DataFrame({'team': active_teams})
strength_table = strength_table.merge(recent_win_pct[['team', 'recent_win_pct']], on='team', how='left')
strength_table = strength_table.merge(hist_win_pct[['team', 'hist_win_pct']], on='team', how='left')

# Fill NaN for new teams with recent_win_pct only (GT, LSG have no pre-2022 history)
strength_table['hist_win_pct'] = strength_table['hist_win_pct'].fillna(strength_table['recent_win_pct'])

# Blended strength: 60% recent + 40% historical
strength_table['strength'] = (0.6 * strength_table['recent_win_pct'] +
                              0.4 * strength_table['hist_win_pct'])
# GT and LSG only exist from 2022 — don't use hist_win_pct for them
new_teams = ['Gujarat Titans', 'Lucknow Super Giants']
strength_table.loc[strength_table['team'].isin(new_teams), 'strength'] = \
    strength_table.loc[strength_table['team'].isin(new_teams), 'recent_win_pct']

strength_table = strength_table.sort_values('strength', ascending=False).reset_index(drop=True)
strength_table['position'] = strength_table.index + 1

print(strength_table[['position', 'team', 'recent_win_pct', 'hist_win_pct', 'strength']])
strength_table = strength_table.sort_values('strength', ascending=False).reset_index(drop=True)
strength_table['position'] = strength_table.index + 1

print(strength_table[['position', 'team', 'recent_win_pct', 'hist_win_pct', 'strength']])

# Top 4
top4 = strength_table.head(4).reset_index(drop=True)
strength_dict = dict(zip(strength_table['team'], strength_table['strength']))

# ── Single playoff simulation ─────────────────────────────────────────────────
print("\n" + "="*50)
print("IPL 2026 PLAYOFF SIMULATION (Based on 2023-2025 Form)")
print("="*50)

t1, t2 = top4.loc[0, 'team'], top4.loc[1, 'team']
q1_winner, p1, p2 = predict_winner(t1, t2, strength_dict)
q1_loser = t2 if q1_winner == t1 else t1
print(f"\nQualifier 1: {t1} vs {t2}")
print(f"  → {t1}: {p1}% | {t2}: {p2}%")
print(f"  → Winner (goes to Final): {q1_winner}")

t3, t4 = top4.loc[2, 'team'], top4.loc[3, 'team']
elim_winner, p3, p4 = predict_winner(t3, t4, strength_dict)
print(f"\nEliminator: {t3} vs {t4}")
print(f"  → {t3}: {p3}% | {t4}: {p4}%")
print(f"  → Winner (survives): {elim_winner}")

q2_winner, p5, p6 = predict_winner(q1_loser, elim_winner, strength_dict)
print(f"\nQualifier 2: {q1_loser} vs {elim_winner}")
print(f"  → {q1_loser}: {p5}% | {elim_winner}: {p6}%")
print(f"  → Winner (goes to Final): {q2_winner}")

final_winner, pf1, pf2 = predict_winner(q1_winner, q2_winner, strength_dict)
print(f"\nFinal: {q1_winner} vs {q2_winner}")
print(f"  → {q1_winner}: {pf1}% | {q2_winner}: {pf2}%")
print(f"\n🏆 PREDICTED IPL 2026 CHAMPION: {final_winner}")

# ── Monte Carlo: 10,000 simulations ──────────────────────────────────────────
from collections import defaultdict
np.random.seed(None)
top4_teams = top4['team'].tolist()
championship_count = defaultdict(int)

for _ in range(10000):
    champion = simulate_playoffs(strength_dict, top4_teams)
    championship_count[champion] += 1

print("\nChampionship Probability (10,000 simulations):")
print("-" * 45)
for team, count in sorted(championship_count.items(),
                           key=lambda x: x[1], reverse=True):
    print(f"{team:35s} {count/10000*100:.1f}%")
# Save
champ_prob = pd.DataFrame([
    {'team': team, 'championship_probability': count/10000*100}
    for team, count in championship_count.items()
]).sort_values('championship_probability', ascending=False)

strength_table.to_csv(rf'{BASE_PATH}\strength_table_2026.csv', index=False)
champ_prob.to_csv(rf'{BASE_PATH}\championship_prob_2026.csv', index=False)
print("\nSaved.")

   position                         team  recent_win_pct  hist_win_pct  \
0         1               Gujarat Titans        0.545455      0.750000   
1         2        Kolkata Knight Riders        0.560976      0.502242   
2         3          Chennai Super Kings        0.500000      0.581731   
3         4  Royal Challengers Bangalore        0.568182      0.464602   
4         5               Mumbai Indians        0.478261      0.554113   
5         6             Rajasthan Royals        0.488372      0.515625   
6         7         Lucknow Super Giants        0.488372      0.600000   
7         8                 Punjab Kings        0.456522      0.458716   
8         9               Delhi Capitals        0.441860      0.459821   
9        10          Sunrisers Hyderabad        0.431818      0.458150   

   strength  
0  0.545455  
1  0.537482  
2  0.532692  
3  0.526750  
4  0.508602  
5  0.499273  
6  0.488372  
7  0.457399  
8  0.449045  
9  0.442351  
   position                    

In [52]:
strength_table.to_csv(rf'{BASE_PATH}\strength_table_2026.csv', index=False)
champ_prob.to_csv(rf'{BASE_PATH}\championship_prob_2026.csv', index=False)
points_table.to_csv(rf'{BASE_PATH}\points_table_2026.csv', index=False)
print("All files saved.")

All files saved.
